# 🧪 Thai Legal A/B Benchmark (Base Typhoon-2 vs Fine-tuned LoRA)
Notebook สำหรับรันประเมินผลเปรียบเทียบหมัดต่อหมัดบนคำถามชุดเดียวกัน เพื่อตอบว่า **LoRA ที่เทรนมาทำให้โมเดลตอบดีกว่า Base ปกติจริงหรือไม่**

> **การตั้งค่า Kaggle:** แถบขวา `Settings` -> `Accelerator` -> **GPU T4 x1** และเปิด **Internet ON**

In [1]:
# 1. ติดตั้ง Dependencies
!pip install -q transformers peft bitsandbytes sentence-transformers rank-bm25 datasets pythainlp accelerate

In [2]:
# 2. โหลด Dataset และสร้าง Corpus Index
from datasets import load_dataset
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

print("📥 Loading WangchanX-Legal-ThaiCCL-RAG dataset...")
ds_train = load_dataset("airesearch/WangchanX-Legal-ThaiCCL-RAG", split="train")
ds_test = load_dataset("airesearch/WangchanX-Legal-ThaiCCL-RAG", split="test")

passages_dict = {}
for split in [ds_train, ds_test]:
    for row in split:
        for ctx in row.get("positive_contexts", []):
            key = ctx.get("unique_key") or ctx.get("context", "")[:50]
            if key not in passages_dict and ctx.get("context", "").strip():
                passages_dict[key] = {
                    "text": ctx["context"],
                    "metadata": ctx.get("metadata", {})
                }

passages = list(passages_dict.values())
print(f"✅ Unique passages indexed: {len(passages)}")

try:
    from pythainlp.tokenize import word_tokenize
    def tokenize(text):
        return word_tokenize(str(text), engine="newmm")
except ImportError:
    def tokenize(text):
        return re.findall(r'[\u0E00-\u0E7F0-9a-zA-Z]+', str(text).lower())

print("🔨 Building BM25 Index...")
tokenized_corpus = [tokenize(p["text"]) for p in passages]
bm25 = BM25Okapi(tokenized_corpus)

print("🔨 Loading Dense Retriever (multilingual-e5-small)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
retriever = SentenceTransformer("intfloat/multilingual-e5-small", device=device)
corpus_texts = [p["text"] for p in passages]
corpus_embeddings = retriever.encode(corpus_texts, convert_to_tensor=True, batch_size=64, show_progress_bar=True)
print("✅ Retrieval indexing completed!")

📥 Loading WangchanX-Legal-ThaiCCL-RAG dataset...


README.md:   0%|          | 0.00/7.01k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.08MB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/72 [00:00<?, ?it/s]

✅ Retrieval indexing completed!


In [3]:
# 3. โหลดโมเดล (Base Model และ Peft Model)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "scb10x/llama3.2-typhoon2-3b"
adapter_repo = "DuckerMaster/thai-legal-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("📦 Loading Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("🔗 Wrapping LoRA Adapter into model...")
model = PeftModel.from_pretrained(base_model, adapter_repo)
model.eval()
print("✅ Setup complete: Can switch between Base and LoRA easily!")

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

...

🔗 Wrapping LoRA Adapter into model...


adapter_config.json:   0%|          | 0.00/911 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 14.7MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✅ Setup complete: Can switch between Base and LoRA easily!


In [4]:
# 4. ฟังก์ชัน Retrieval และ Generator สำหรับ Base vs LoRA

def hybrid_retrieve(query, top_k=1):
    q_tokens = tokenize(query)
    bm25_scores = np.array(bm25.get_scores(q_tokens))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()
        
    q_emb = retriever.encode(query, convert_to_tensor=True)
    dense_sim = torch.cosine_similarity(q_emb, corpus_embeddings).cpu().numpy()
    combined = 0.5 * dense_sim + 0.5 * bm25_scores
    top_idx = np.argsort(combined)[-top_k:][::-1]
    return [passages[i] for i in top_idx]

def generate_prediction(question, context, use_adapter=True):
    prompt = f"จากกฎหมายต่อไปนี้:\n{context}\n\nคำถาม: {question}\nคำตอบ:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # สลับระหว่าง LoRA กับ Base โดยใช้ context manager ของ PEFT
    context_mgr = model.disable_adapter() if not use_adapter else torch.no_grad()
    with context_mgr:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "คำตอบ:" in full_text:
        return full_text.split("คำตอบ:")[-1].strip()
    return full_text[len(prompt):].strip()

In [5]:
# 5. ฟังก์ชันคำนวณ Metrics
def _lcs(x, y):
    m, n = len(x), len(y)
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            curr[j] = prev[j - 1] + 1 if x[i - 1] == y[j - 1] else max(curr[j - 1], prev[j])
        prev = curr
    return prev[n]

def rouge_l(pred, ref):
    pred_t = re.findall(r'[\u0E00-\u0E7F0-9a-zA-Z]+', pred.lower())
    ref_t = re.findall(r'[\u0E00-\u0E7F0-9a-zA-Z]+', ref.lower())
    if not pred_t or not ref_t: return 0.0
    l = _lcs(pred_t, ref_t)
    p, r = l / len(pred_t), l / len(ref_t)
    return (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

def extract_sections(text):
    return set(re.findall(r'มาตรา\s*(\d+(?:/\d+)?)', text))

def section_match(pred, gt_secs):
    if not gt_secs: return True
    return len(extract_sections(pred) & gt_secs) > 0

In [6]:
# 6. เริ่มการทดสอบ A/B Testing (Base vs LoRA) 50 ตัวอย่าง
import random, time, json

random.seed(42)
test_samples = [r for r in ds_test if r.get("positive_answer", "").strip()]
samples = random.sample(test_samples, min(50, len(test_samples)))
print(f"🚀 Starting A/B Comparison on {len(samples)} samples...\n")

base_rouge, lora_rouge = [], []
base_sec_hits, lora_sec_hits = 0, 0
valid_sec_checks = 0
comparisons = []

for i, item in enumerate(samples, 1):
    q = item["question"]
    gt = item["positive_answer"]
    
    gt_secs = set()
    for ctx in item.get("positive_contexts", []):
        s = ctx.get("metadata", {}).get("section", "")
        if s: gt_secs.add(str(s))
        gt_secs |= extract_sections(ctx.get("context", ""))
    
    retrieved = hybrid_retrieve(q, top_k=1)
    ctx_text = retrieved[0]["text"]
    
    # 1. Base Model Answer
    pred_base = generate_prediction(q, ctx_text, use_adapter=False)
    rl_base = rouge_l(pred_base, gt)
    base_rouge.append(rl_base)
    
    # 2. LoRA Model Answer
    pred_lora = generate_prediction(q, ctx_text, use_adapter=True)
    rl_lora = rouge_l(pred_lora, gt)
    lora_rouge.append(rl_lora)
    
    hit_base, hit_lora = False, False
    if gt_secs:
        valid_sec_checks += 1
        if section_match(pred_base, gt_secs):
            base_sec_hits += 1
            hit_base = True
        if section_match(pred_lora, gt_secs):
            lora_sec_hits += 1
            hit_lora = True
            
    comparisons.append({
        "id": i,
        "question": q,
        "ground_truth": gt,
        "base_pred": pred_base,
        "lora_pred": pred_lora,
        "base_rouge_l": rl_base,
        "lora_rouge_l": rl_lora,
        "base_sec_hit": hit_base,
        "lora_sec_hit": hit_lora
    })
    print(f"[{i:02d}/50] Base ROUGE: {rl_base:.3f} | LoRA ROUGE: {rl_lora:.3f} | Diff: {rl_lora - rl_base:+.3f}")

# สรุปผล A/B Testing
avg_b_rouge = sum(base_rouge) / len(base_rouge)
avg_l_rouge = sum(lora_rouge) / len(lora_rouge)
b_sec_rate = base_sec_hits / valid_sec_checks if valid_sec_checks > 0 else 0
l_sec_rate = lora_sec_hits / valid_sec_checks if valid_sec_checks > 0 else 0

print("\n" + "="*60)
print("📊 A/B BENCHMARK RESULT (Base Model vs LoRA Fine-tuned)")
print("="*60)
print(f"Metric                  | Base Typhoon-2 (3B) | LoRA Fine-tuned     | Delta")
print(f"ROUGE-L (avg)           | {avg_b_rouge:.4f} ({avg_b_rouge*100:.2f}%)   | {avg_l_rouge:.4f} ({avg_l_rouge*100:.2f}%)   | {avg_l_rouge - avg_b_rouge:+.4f}")
print(f"Section Match Rate      | {b_sec_rate:.2%} ({base_sec_hits}/{valid_sec_checks})     | {l_sec_rate:.2%} ({lora_sec_hits}/{valid_sec_checks})     | {l_sec_rate - b_sec_rate:+.2%}")
print("="*60)

with open("ab_eval_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "base_rouge_l": avg_b_rouge,
            "lora_rouge_l": avg_l_rouge,
            "base_section_rate": b_sec_rate,
            "lora_section_rate": l_sec_rate
        },
        "comparisons": comparisons
    }, f, ensure_ascii=False, indent=2)
print("💾 Detailed A/B comparisons saved to ab_eval_results.json")

🚀 Starting A/B Comparison on 50 samples...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[01/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.741 | Diff: +0.741
[02/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.095 | Diff: +0.095
[03/50] Base ROUGE: 0.222 | LoRA ROUGE: 0.600 | Diff: +0.378
[04/50] Base ROUGE: 0.087 | LoRA ROUGE: 0.107 | Diff: +0.020
[05/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.000 | Diff: +0.000
[06/50] Base ROUGE: 0.174 | LoRA ROUGE: 0.250 | Diff: +0.076
[07/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.174 | Diff: +0.174
[08/50] Base ROUGE: 0.061 | LoRA ROUGE: 0.857 | Diff: +0.797
[09/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.000 | Diff: +0.000
[10/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.526 | Diff: +0.526
[11/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.065 | Diff: +0.065
[12/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.071 | Diff: +0.071
[13/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.387 | Diff: +0.387
[14/50] Base ROUGE: 0.276 | LoRA ROUGE: 0.424 | Diff: +0.148
[15/50] Base ROUGE: 0.000 | LoRA ROUGE: 0.792 | Diff: +0.792
[16/50] Base ROUGE: 0.069 | LoRA ROUGE: 0.061 | Diff: -0.008
[17/50] Base ROUGE: 0.00